# Thử sinh mô tả chi tiết bằng Qwen2.5-VL — bản v2 (20 ảnh)

**Sửa so với v1** (v1 chép ví dụ trong câu lệnh: gần như ảnh nào cũng "có khăn cam trên vòi, thớt đỏ, 2 thùng rác"):
1. Câu lệnh **không còn ví dụ cụ thể** — chỉ có khung trống, bắt ghi `none` khi không thấy.
2. **Tả từng người trên ảnh cắt riêng** theo khung YOLO có sẵn (`khung_nguoi.json`) → không bịa thêm / lặp người.
3. **Tả đồ vật trên vùng cắt phóng to**: đảo bếp + bồn rửa, bếp nấu, phòng khách / sofa.
4. Thêm 1 lượt tả cả ảnh (đếm người, vị trí trong khung hình).
5. Tự chấm: (a) 4 ảnh đáp án có đủ chi tiết của đề không, (b) **ảnh nào trong 20 ảnh khớp đủ chi tiết của từng đề**
   — nếu chỉ ảnh đáp án (và vài ảnh liền kề) khớp thì mô tả đủ tốt để tìm kiếm, (c) kiểm tra còn "chép" không.

**Cách chạy:** Runtime → Change runtime type → **L4** (hoặc A100). T4: xem ghi chú ở ô 4.
1. Chạy ô 1, ô 2 (cài đặt), rồi **Runtime → Restart session** (bắt buộc, vì vừa đổi phiên bản thư viện).
2. Bấm vào ô **2b** → menu **Runtime → Run cell and below** (chạy từ đó tới hết, không cần ngồi canh).

**Tự ngắt phiên để không tốn compute unit:** từ ô 2b trở đi, phiên GPU **tự ngắt** khi (a) chạy xong, (b) có ô bị lỗi,
hoặc (c) chạy quá `GIOI_HAN_GIO` giờ (treo). Trước khi ngắt, kết quả + nhật ký lỗi được lưu vào Google Drive:
`MyDrive/image_retrieval_colab/` (`mo_ta_thu_20_v2.jsonl`, `toc_do.txt`, `nhat_ky.txt`). Bấm nút Stop (dừng tay) thì
KHÔNG tự ngắt. Lấy 2 file kết quả từ Drive và gửi lại.

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Cài đặt (~3–5 phút). GHIM PHIÊN BẢN: vLLM mới nhất (0.34) lỗi khi khởi động vì thiếu gói "cutlass"
#    (module của model MiniMax, không liên quan Qwen) -> dùng bản ổn định đã hỗ trợ Qwen2.5-VL + transformers khớp.
!pip install -q "vllm==0.10.2" "transformers==4.56.2" qwen-vl-utils
import importlib.metadata as _md
print({p: _md.version(p) for p in ("vllm", "transformers", "torch")})
# BẮT BUỘC: Runtime -> Restart session (phiên bản torch/transformers vừa đổi), rồi bấm ô 2b -> Run cell and below.

In [ ]:
# 2b. BẢO VỆ PHIÊN: lưu kết quả vào Google Drive và TỰ NGẮT phiên GPU khi xong / khi lỗi / khi quá giờ.
import os, time, shutil, threading, traceback
from google.colab import drive, runtime
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/image_retrieval_colab"
os.makedirs(OUT, exist_ok=True)
KET_QUA_FILES = ["mo_ta_thu_20_v2.jsonl", "toc_do.txt"]
GIOI_HAN_GIO = 2          # bản thử 20 ảnh chỉ cần ~15–30 phút; quá 2 giờ coi như bị treo -> ngắt
TU_NGAT = True            # đặt False nếu muốn giữ phiên để xem / chạy lại bằng tay
_da_ngat = False

def ghi_nhat_ky(dong):
    with open(f"{OUT}/nhat_ky.txt", "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {dong}\n")

def luu_ket_qua():
    for fn in KET_QUA_FILES:
        if os.path.exists(fn):
            shutil.copy(fn, OUT)

def ngat(ly_do):
    global _da_ngat
    if _da_ngat:
        return
    _da_ngat = True
    try:
        ghi_nhat_ky(f"NGẮT PHIÊN: {ly_do}")
        luu_ket_qua()
        drive.flush_and_unmount()          # đảm bảo file đã ghi xong lên Drive
        print(f"Đã lưu vào {OUT}. Ngắt phiên: {ly_do}")
    finally:
        if TU_NGAT:
            runtime.unassign()             # trả GPU, dừng tính compute unit

def _khi_loi(shell, etype, evalue, tb, tb_offset=None):
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    ghi_nhat_ky("LỖI:\n" + "".join(traceback.format_exception(etype, evalue, tb))[-4000:])
    ngat(f"lỗi {etype.__name__}: {evalue}")

# Chỉ bắt Exception: bấm Stop (KeyboardInterrupt) thì không ngắt phiên.
get_ipython().set_custom_exc((Exception,), _khi_loi)
_hen_gio = threading.Timer(GIOI_HAN_GIO * 3600, lambda: ngat(f"quá {GIOI_HAN_GIO} giờ (có thể bị treo)"))
_hen_gio.daemon = True
_hen_gio.start()
ghi_nhat_ky("BẮT ĐẦU")
print(f"Tự ngắt: bật (xong / lỗi / quá {GIOI_HAN_GIO} giờ). Kết quả lưu ở {OUT}")

In [ ]:
# 3. Tải lên anh_thu_20_v2.zip (trong thư mục colab/ của repo) rồi giải nén
import os, zipfile, csv, json
from google.colab import files
if not os.path.exists("anh_thu_20_v2.zip"):
    files.upload()
zipfile.ZipFile("anh_thu_20_v2.zip").extractall("anh_thu")
loai = {r["file"]: r["loai"] for r in csv.DictReader(open("anh_thu/danh_sach.csv", encoding="utf-8"))}
khung = json.load(open("anh_thu/khung_nguoi.json"))
anh = sorted(loai)
print(len(anh), "ảnh,", sum(len(v) for v in khung.values()), "người (khung YOLO)")

In [ ]:
# 4. Nạp model. Thử vLLM trước (nhanh); lỗi thì tự chuyển sang transformers (chậm hơn, cùng model).
#    GPU T4 (16 GB, không có bfloat16): đổi MODEL sang "Qwen/Qwen2.5-VL-3B-Instruct" và DTYPE = "float16".
#    Báo hết bộ nhớ GPU sau khi vLLM lỗi: kết nối lại, đặt DUNG_VLLM = False, chạy lại từ ô 2b.
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
DTYPE = "bfloat16"
DUNG_VLLM = True
MAX_PIXELS = 1280 * 720
llm = None
if DUNG_VLLM:
    try:
        from vllm import LLM, SamplingParams
        llm = LLM(model=MODEL, dtype=DTYPE, max_model_len=8192, gpu_memory_utilization=0.90,
                  limit_mm_per_prompt={"image": 1},
                  mm_processor_kwargs={"min_pixels": 64 * 28 * 28, "max_pixels": MAX_PIXELS})
        import vllm
        print("Đang dùng vLLM", vllm.__version__)
    except Exception as e:
        print("vLLM lỗi, chuyển sang transformers (CHẬM ~30 giây/ảnh):", repr(e)[:300])
        ghi_nhat_ky("vLLM lỗi: " + repr(e)[:1500])
        llm = None
if llm is None:
    import gc, torch
    gc.collect(); torch.cuda.empty_cache()
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    hf_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL, torch_dtype=getattr(torch, DTYPE), device_map="cuda", attn_implementation="sdpa").eval()
    hf_proc = AutoProcessor.from_pretrained(MODEL, min_pixels=64 * 28 * 28, max_pixels=MAX_PIXELS)
    hf_proc.tokenizer.padding_side = "left"
    print("Đang dùng transformers")

In [ ]:
# 5. Câu lệnh — KHÔNG có ví dụ cụ thể (v1 bị model chép ví dụ). Chỉ khung trống + bắt ghi "none".
NGUOI = """This image is a crop of ONE person from an overhead fisheye kitchen camera (the person may look small or
distorted; parts of other people may appear at the edges - ignore them). Describe only the main person, only what is
clearly visible. If something is not visible or you are not sure, write "none". Do not guess.
Replace every <...> placeholder with your own words; never output the angle brackets themselves.
Return valid JSON only:
{"gender": "...", "hair": "...", "top": "<colour> <garment type> <pattern if any>", "bottom": "...",
 "headwear": "...", "eyewear": "...", "other_accessories": "...",
 "holding": "<object(s) in the hands with colour, or none>", "action": "...",
 "posture": "standing | sitting | bending | walking", "facing": "toward camera | back to camera | sideways"}"""

VUNG = {
    "island_sink": ("the kitchen island with its sink and faucet", (0.20, 0.20, 0.66, 0.66)),
    "stove":       ("the stove area", (0.50, 0.08, 0.80, 0.58)),
    "living_room": ("the living room with the sofa", (0.03, 0.08, 0.36, 0.50)),
}
def vung_prompt(ten):
    return f"""This image is a zoomed crop of {ten} from an overhead kitchen camera. List only what you can clearly
see. Do not assume typical kitchen items. If nothing is there, use an empty list.
List each object at most once and at most 12 objects. Replace the <...> placeholders with your own words; never output
the angle brackets themselves.
Return valid JSON only:
{{"objects": ["<colour> <object> <where exactly, e.g. on/next to/hanging from what>", ...],
 "people": ["<clothing, headwear, hair, posture, action>", ...]}}"""

CA_ANH = """This is a full frame from a fixed overhead fisheye camera in a home kitchen with a living room in the back.
Count every visible person, including small or partly hidden people in the back. Describe only what is visible; do not
guess. Return valid JSON only:
{"people_count": <integer>,
 "people": [{"where_in_room": "...", "where_in_frame": "<e.g. bottom-left, center, top-right>", "short": "<clothing + action>"}],
 "caption": "<2-4 sentences mentioning every person and any small coloured objects you can clearly see>"}"""

In [ ]:
# 6. Tạo các yêu cầu: mỗi người (cắt theo khung YOLO, nới 15%), mỗi vùng, cả ảnh -> chạy 1 lô.
from PIL import Image
def cat_anh(im, box, pad=0.0, min_side=224):
    W, H = im.size
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    x1, x2 = max(0, x1 - pad * w), min(1, x2 + pad * w)
    y1, y2 = max(0, y1 - pad * h), min(1, y2 + pad * h)
    c = im.crop((int(x1 * W), int(y1 * H), int(x2 * W), int(y2 * H)))
    s = min_side / max(1, min(c.size))
    if s > 1:                                    # phóng to người nhỏ để model nhìn rõ
        c = c.resize((int(c.width * s), int(c.height * s)), Image.BICUBIC)
    return c

yeu_cau = []                                     # (ảnh, loại, khoá, PIL image, câu lệnh)
for f in anh:
    im = Image.open(f"anh_thu/{f}").convert("RGB")
    for k, b in enumerate(khung[f]):
        yeu_cau.append((f, "nguoi", f"p{k}_conf{b[4]}", cat_anh(im, b[:4], pad=0.15), NGUOI))
    for ten, (mo_ta, box) in VUNG.items():
        yeu_cau.append((f, "vung", ten, cat_anh(im, box), vung_prompt(mo_ta)))
    yeu_cau.append((f, "ca_anh", "full", im, CA_ANH))
print(len(yeu_cau), "yêu cầu")

In [ ]:
# 7. Chạy. temperature=0: luôn chọn từ chắc nhất (không bịa, chạy lại ra cùng kết quả).
import base64, io, re, time
def b64(img):
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=92)
    return base64.b64encode(buf.getvalue()).decode()

def chay_transformers(ds, lo=6):
    import torch
    kq = []
    for i in range(0, len(ds), lo):
        batch = ds[i:i + lo]
        texts = [hf_proc.apply_chat_template([{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": p}]}],
                                             tokenize=False, add_generation_prompt=True) for _, p in batch]
        inp = hf_proc(text=texts, images=[img for img, _ in batch], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            g = hf_model.generate(**inp, max_new_tokens=700, do_sample=False, repetition_penalty=1.05)
        kq += hf_proc.batch_decode(g[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"  xong {min(i + lo, len(ds))}/{len(ds)}")
    return kq

t = time.time()
if llm is not None:
    # mỗi loại yêu cầu 1 giới hạn độ dài; repetition_penalty chặn lỗi lặp "pineapple on countertop" hàng trăm lần (v2)
    TOI_DA = {"nguoi": 300, "vung": 500, "ca_anh": 600}
    sps = [SamplingParams(temperature=0, max_tokens=TOI_DA[kind], repetition_penalty=1.05) for _, kind, _, _, _ in yeu_cau]
    convs = [[{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64(img)}"}},
        {"type": "text", "text": p}]}] for _, _, _, img, p in yeu_cau]
    van_ban = [o.outputs[0].text for o in llm.chat(convs, sps)]
else:
    van_ban = chay_transformers([(img, p) for _, _, _, img, p in yeu_cau])
giay = time.time() - t
cach = "vLLM" if llm is not None else "transformers"
print(f"{len(yeu_cau)} yêu cầu / {len(anh)} ảnh trong {giay:.0f} giây = {giay / len(anh):.1f} giây/ảnh ({cach})")
print(f"Ước lượng 5000 ảnh với cách này: ~{giay / len(anh) * 5000 / 3600:.1f} giờ")

def doc_json(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

ket_qua = {f: {"file": f, "loai": loai[f], "nguoi": [], "vung": {}, "ca_anh": None, "loi_json": 0} for f in anh}
for (f, kind, key, _, _), txt in zip(yeu_cau, van_ban):
    js = doc_json(txt)
    r = ket_qua[f]
    if js is None:
        r["loi_json"] += 1
        js = {"raw": txt}
    if kind == "nguoi":
        r["nguoi"].append({"id": key, **js})
    elif kind == "vung":
        r["vung"][key] = js
    else:
        r["ca_anh"] = js
with open("mo_ta_thu_20_v2.jsonl", "w", encoding="utf-8") as fh:
    for f in anh:
        fh.write(json.dumps(ket_qua[f], ensure_ascii=False) + "\n")
fh = open("toc_do.txt", "w", encoding="utf-8"); fh.write(f"{giay:.1f} giây cho {len(anh)} ảnh, {len(yeu_cau)} yêu cầu, {cach}, {MODEL}\n"); fh.close()
print("JSON lỗi:", sum(r["loi_json"] for r in ket_qua.values()), "/", len(yeu_cau))

In [ ]:
# 8. Tự chấm.
# (a) 4 ảnh đáp án có nhắc đủ chi tiết của đề không   (b) ẢNH NÀO trong 20 ảnh khớp đủ chi tiết của từng đề
# (c) còn "chép" không: chi tiết nào xuất hiện ở quá nhiều ảnh thì vô dụng để tìm.
DE = {
    "02 (18_939)": {"dap_an": "18_939.jpg", "so_nguoi": 1, "chi_tiet": {
        "điện thoại": r"phone", "áo len kem": r"(cream|beige|off-white|ivory)[^\"]*(sweater|jumper|knit)|sweater[^\"]*(cream|beige)",
        "váy đen": r"black (skirt|dress)", "tóc dài": r"long[^\"]*hair"}},
    "03 (16_2624)": {"dap_an": "16_2624.jpg", "so_nguoi": 5, "chi_tiet": {
        "áo thể thao xanh": r"(blue|navy)[^\"]*(jersey|sports)", "cầm thớt": r"(cutting|chopping) board",
        "áo sọc ngang": r"strip", "quay lưng": r"quaylung"}},
    "04 (12_320)": {"dap_an": "12_320.jpg", "so_nguoi": 3, "chi_tiet": {
        "rửa tay": r"wash\w*[^\"]*hands", "áo sơ mi xanh": r"blue[^\"]*shirt",
        "camera trên đầu": r"head[- ]?mounted|camera on (his |the )?(forehead|head)|headlamp|head lamp|headband|goggles",
        "khăn cam": r"orange[^\"]*(towel|cloth|rag)", "người ngồi sofa": r"sit\w*[^\".]*(sofa|couch)|(sofa|couch)[^\".]*sit", "áo trắng": r"white[^\"]*(shirt|top|t-shirt)",
        "mũ lưỡi trai": r"\bcap\b|baseball", "tóc xoăn": r"curly"}},
    "05 (15_2827)": {"dap_an": "15_2827.jpg", "so_nguoi": 1, "chi_tiet": {
        "điện thoại": r"phone", "đeo kính": r"glasses|spectacles", "áo len xanh đậm": r"(navy|dark blue)[^\"]*(sweater|jumper|knit)",
        "góc dưới trái": r"bottom[- ]left|lower[- ]left"}},
}
# đổi "back to camera" -> "quaylung", "toward camera" -> "huongve" để chữ "camera" (camera trên trán) không khớp nhầm
def _chuan(t):
    t = re.sub(r"(back to|away from) (the )?camera", "quaylung", t)
    return re.sub(r"(toward|towards|facing) (the )?camera", "huongve", t)
chu = {f: _chuan(json.dumps(r, ensure_ascii=False).lower()) for f, r in ket_qua.items()}
for ten, de in DE.items():
    f = de["dap_an"]; r = ket_qua[f]
    n = (r["ca_anh"] or {}).get("people_count")
    print(f"\n=== Đề {ten}: số người model đếm {n} (thật {de['so_nguoi']}), YOLO {len(khung[f])}")
    for ct, pat in de["chi_tiet"].items():
        co = [g for g in anh if re.search(pat, chu[g])]
        print(f"   {'✓' if f in co else '✗'} {ct:18s} có ở {len(co):2d}/20 ảnh")
    diem = {g: sum(bool(re.search(p, chu[g])) for p in de["chi_tiet"].values()) for g in anh}
    xep = sorted(anh, key=lambda g: -diem[g])
    print("   Ảnh khớp nhiều chi tiết nhất:", ", ".join(f"{g[:-4]}({diem[g]}/{len(de['chi_tiet'])}{'*' if g == f else ''})" for g in xep[:6]),
          " — * = đáp án")

In [ ]:
# 9. Xem chi tiết từng ảnh (đọc kỹ 4 ảnh đáp án và ảnh liền kề / ảnh đã nộp sai)
from IPython.display import Image as IPImage, display
for f in anh:
    r = ket_qua[f]
    print(f"\n#### {f}  [{r['loai']}]")
    display(IPImage(f"anh_thu/{f}", width=480))
    print("CẢ ẢNH:", json.dumps(r["ca_anh"], ensure_ascii=False))
    for p in r["nguoi"]:
        print("  NGƯỜI", json.dumps(p, ensure_ascii=False))
    for k, v in r["vung"].items():
        print(f"  VÙNG {k}:", json.dumps(v, ensure_ascii=False))

In [ ]:
# 10. Lưu kết quả vào Drive (MyDrive/image_retrieval_colab/) rồi TỰ NGẮT phiên GPU.
#     (Không dùng tải về trình duyệt: lệnh tải chạy ngầm, ngắt phiên ngay sau đó có thể làm mất file.)
_hen_gio.cancel()
ngat("chạy xong")